# CoLight Training on Google Colab
**Thesis:** Distributed Control of Multi-Agent Systems using Reinforcement Learning  
**Author:** Ahmed Elafandy, GUC Mechatronics Engineering  

**Before running:** Make sure you have selected a GPU runtime:  
Runtime → Change runtime type → T4 GPU

In [1]:
# ─── CELL 1: Verify GPU ───────────────────────────────────────────────────────
!nvidia-smi
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [7]:
# ─── CELL 2: Clone CoLight repo ──────────────────────────────────────────────
import os

GITHUB_USERNAME = 'flameplat'   # ← change this
REPO_NAME       = 'colight-Bachelor'  # ← change this

if not os.path.exists(f'/content/{REPO_NAME}'):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    print('Repo already cloned, pulling latest...')
    !cd /content/{REPO_NAME} && git pull

%cd /content/{REPO_NAME}
!ls

In [ ]:
# ─── CELL 2.5: Keep-alive (prevent Colab inactivity timeout) ─────────────────
import threading, time

def _keep_alive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive thread started")

In [3]:
# ─── CELL 3: Install Python dependencies ─────────────────────────────────────
# NOTE: Do NOT install tensorflow — Colab already has it with CUDA support
# Do NOT install tensorflow-macos or tensorflow-metal — those are macOS only

!pip install -q networkx scipy pandas matplotlib

# Verify keras version bundled with Colab's TF
import keras
print('Keras version:', keras.__version__)

In [4]:
# ─── CELL 4: Build CityFlow from source ───────────────────────────────────────
!apt-get install -q -y cmake build-essential python3-dev
!git submodule update --init --recursive
!pip install -q /content/{REPO_NAME}/CityFlow

# Verify
import cityflow
print('CityFlow OK')


In [5]:
# ─── CELL 5: Verify full environment ─────────────────────────────────────────
import tensorflow as tf
import keras
import cityflow
import networkx
import scipy

print('TF version    :', tf.__version__)
print('Keras version :', keras.__version__)
print('GPU available :', tf.config.list_physical_devices('GPU'))
print('CityFlow      : OK')
print('NetworkX      :', networkx.__version__)
print('SciPy         :', scipy.__version__)

In [10]:
# ─── CELL 6: Quick test run (3x3, short) ─────────────────────────────────────
# Run this first to confirm everything works before committing to full experiment

%cd /content/{REPO_NAME}

!python runexp.py \
    --memo test_colab \
    --env 1 \
    --road_net 3_3 \
    --volume 300 \
    --suffix 0.3 \
    --mod CoLight \
    --cnt 360 \
    --gen 1 \
    --workers 1

In [ ]:
# ─── CELL 7a: Mount Drive + start checkpoint thread ──────────────────────────
# Run this BEFORE Cell 7 for a fresh training run.
# Syncs model/ and records/ to Drive every 10 min so crashes don't lose progress.

from google.colab import drive
drive.mount('/content/drive')

import threading, shutil, os, time

REPO       = f'/content/{REPO_NAME}'
DRIVE_SAVE = '/content/drive/MyDrive/colight_results'
CKPT_EVERY = 10 * 60  # seconds between auto-syncs

os.makedirs(DRIVE_SAVE, exist_ok=True)

_ckpt_stop = threading.Event()

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    for sub in ['model', 'records']:
        src = os.path.join(REPO, sub)
        dst = os.path.join(DRIVE_SAVE, sub)
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"[{ts}] Checkpoint '{label}' saved to Drive")

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f"Checkpoint thread started — syncing every {CKPT_EVERY // 60} min to {DRIVE_SAVE}")

In [ ]:
# ─── CELL 7: Full experiment — CoLight, 6x6, 150 rounds ──────────────────────
# Run after Cell 7a. Cell 7a must be run first to start the checkpoint thread.

%cd /content/{REPO_NAME}

try:
    !python runexp.py \
        --memo topology_colight_6_6_900_grid \
        --road_net 6_6 \
        --volume 900 \
        --suffix 0.3_turn_drain \
        --topology optimal \
        --rounds 150 \
        2>&1 | tee /content/train_log.txt
finally:
    _ckpt_stop.set()
    _ckpt_thread.join(timeout=5)
    _do_checkpoint("final")

In [ ]:
# ─── CELL 7b: RESUME — restore from Drive and locate checkpoint ───────────────
# Run Cells 1–5 first, then run THIS cell instead of Cell 7a + Cell 7.

from google.colab import drive
drive.mount('/content/drive')

import threading, shutil, os, time, json, glob

REPO       = f'/content/{REPO_NAME}'
DRIVE_SAVE = '/content/drive/MyDrive/colight_results'
CKPT_EVERY = 10 * 60

# Restore model/ and records/ from Drive back into the VM
for sub in ['model', 'records']:
    src = os.path.join(DRIVE_SAVE, sub)
    dst = os.path.join(REPO, sub)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Restored {sub}/')
    else:
        print(f'WARNING: {src} not found on Drive')

# Find the most recent checkpoint.json
ckpt_files = glob.glob(f'{REPO}/model/**/checkpoint.json', recursive=True)
if not ckpt_files:
    raise RuntimeError('No checkpoint.json found — nothing to resume.')

ckpt_path   = max(ckpt_files, key=os.path.getmtime)
ckpt        = json.load(open(ckpt_path))
RESUME_PATH = ckpt['model_dir']
last_round  = ckpt['last_completed_round']
print(f'Found checkpoint: round {last_round}')
print(f'Model dir: {RESUME_PATH}')
print(f'Will resume from round {last_round + 1}')

# Start checkpoint thread for the resumed run
os.makedirs(DRIVE_SAVE, exist_ok=True)
_ckpt_stop = threading.Event()

def _do_checkpoint(label='auto'):
    ts = time.strftime('%H:%M:%S')
    for sub in ['model', 'records']:
        src = os.path.join(REPO, sub)
        dst = os.path.join(DRIVE_SAVE, sub)
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"[{ts}] Checkpoint '{label}' saved to Drive")

def _ckpt_loop():
    while not _ckpt_stop.wait(timeout=CKPT_EVERY):
        _do_checkpoint()

_ckpt_thread = threading.Thread(target=_ckpt_loop, daemon=True)
_ckpt_thread.start()
print(f'Checkpoint thread started — syncing every {CKPT_EVERY // 60} min')

In [ ]:
# ─── CELL 7c: RESUME — continue training from checkpoint ─────────────────────
# Run after Cell 7b. RESUME_PATH is set by Cell 7b automatically.

%cd /content/{REPO_NAME}

try:
    !python runexp.py \
        --memo topology_colight_6_6_900_grid \
        --road_net 6_6 \
        --volume 900 \
        --suffix 0.3_turn_drain \
        --topology optimal \
        --rounds 150 \
        --resume {RESUME_PATH} \
        2>&1 | tee /content/train_log_resume.txt
finally:
    _ckpt_stop.set()
    _ckpt_thread.join(timeout=5)
    _do_checkpoint("final")

In [ ]:
# ─── CELL 8: Save results to Google Drive ────────────────────────────────────
# Run after training to back up results before Colab session ends

from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

SAVE_DIR = '/content/drive/MyDrive/colight_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy model checkpoints
if os.path.exists(f'/content/{REPO_NAME}/model'):
    shutil.copytree(
        f'/content/{REPO_NAME}/model',
        f'{SAVE_DIR}/model',
        dirs_exist_ok=True)
    print('Models saved to Drive')

# Copy training records
if os.path.exists(f'/content/{REPO_NAME}/records'):
    shutil.copytree(
        f'/content/{REPO_NAME}/records',
        f'{SAVE_DIR}/records',
        dirs_exist_ok=True)
    print('Records saved to Drive')

print('All results backed up to Google Drive')